In [17]:
def flatten_posts_dict(posts_by_tag):
    flat_posts = {}
    for tag, posts in posts_by_tag.items():
        for uri, post in posts.items():
            post = dict(post)          
            post["hashtag"] = tag      
            flat_posts[uri] = post
    return flat_posts


def add_reposters_to_posts(posts_dict, users):
    for post in posts_dict.values():
        post["reposted_by"] = []

    for did, user in users.items():
        for uri in user.get("reposted_posts", []):
            if uri in posts_dict:
                posts_dict[uri]["reposted_by"].append(did)

    for post in posts_dict.values():
        post["repost_user_unobservable"] = (
            post.get("repostCount", 0) > 0 and not post["reposted_by"]
        )

In [18]:
import json

with open("TEST_X2.json", "r", encoding="utf-8") as f:
    raw_user_data = json.load(f)

with open("posts.json", "r", encoding="utf-8") as f:
    posts = json.load(f)

post_dict = flatten_posts_dict(posts)
add_reposters_to_posts(post_dict, raw_user_data)

In [20]:
import pandas as pd
import numpy as np
from datetime import datetime, timezone

# =====================================================
# Time utilities (timezone-safe)
# =====================================================

def parse_dt(ts):
    if not ts:
        return None
    try:
        return datetime.fromisoformat(ts.replace("Z", "+00:00")).astimezone(timezone.utc)
    except Exception:
        return None


# =====================================================
# History-based helpers
# =====================================================

def history_stats(history):
    if not history:
        return 0, 0.0, None, None

    reposts = 0
    engagements = []
    times = []

    for h in history:
        if h.get("activity_type") == "repost":
            reposts += 1

        if h.get("created_at"):
            t = parse_dt(h["created_at"])
            if t:
                times.append(t)

        for k in ("like_count", "repost_count", "reply_count", "quote_count"):
            if h.get(k) is not None:
                engagements.append(h[k])

    times.sort()
    gaps = [(times[i] - times[i - 1]).days for i in range(1, len(times))]

    return (
        len(history),
        reposts / len(history),
        np.mean(gaps) if gaps else None,
        np.mean(engagements) if engagements else None,
    )


def mention_stats(history, handle):
    if not history or not handle:
        return 0, 0.0

    total = len(history)
    count = sum(handle in (h.get("text") or "") for h in history)
    return count, count / total


def reposts_from_author_before(history, author_did, post_time):
    if not history or not post_time:
        return 0

    cnt = 0
    for h in history:
        if h.get("activity_type") != "repost":
            continue
        if h.get("post_author_did") != author_did:
            continue

        t = parse_dt(h.get("reposted_at"))
        if t and t < post_time:
            cnt += 1

    return cnt


# =====================================================
# User feature extraction
# =====================================================

def user_features(user, max_posts):
    age = user["stats"]["account_age_days"] or 1
    posts = user["stats"]["posts"] or 0

    hist_n, repost_pct, avg_gap, avg_eng = history_stats(user["history"])

    return {
        "age_days": age,
        "followers": user["stats"]["followers"],
        "followees": user["stats"]["follows"],
        "total_posts": posts,
        "posts_ratio": posts / max_posts,
        "followers_per_day": user["stats"]["followers"] / age,
        "followees_per_day": user["stats"]["follows"] / age,
        "posts_per_day": posts / age,
        "profile_has_url": int("http" in (user["profile"]["description"] or "")),
        "hist_msgs": hist_n,
        "hist_repost_pct": repost_pct,
        "hist_avg_gap_days": avg_gap,
        "hist_avg_engagement": avg_eng,
    }


# =====================================================
# FINAL DATASET BUILDER (SCHEMA-LOCKED)
# =====================================================

def build_repost_triples_df(
    users,
    posts_dict,
    n_posts_sample=None,
    neg_per_pos=5,
    seed=42,
):
    rng = np.random.default_rng(seed)
    all_users = set(users.keys())

    post_items = list(posts_dict.items())
    if n_posts_sample is not None:
        post_items = rng.choice(
            post_items,
            size=min(n_posts_sample, len(post_items)),
            replace=False,
        )

    max_posts = max(u["stats"]["posts"] or 1 for u in users.values())

    rows = []

    for P_id, post in post_items:
        S_id = post.get("author", {}).get("did")
        if not S_id or S_id not in users:
            continue

        reposted_by = set(post.get("reposted_by", []))
        if not reposted_by:
            continue

        P_time = parse_dt(post.get("indexedAt"))
        hashtag = post.get("hashtag")

        S = users[S_id]
        S_feat = user_features(S, max_posts)

        candidate_users = all_users - {S_id}

        # ================= POSITIVES =================
        for A_id in reposted_by:
            if A_id not in users:
                continue

            A = users[A_id]
            A_feat = user_features(A, max_posts)

            A_m_S, A_m_S_per = mention_stats(A["history"], S["profile"]["handle"])
            S_m_A, S_m_A_per = mention_stats(S["history"], A["profile"]["handle"])

            rows.append({
                # IDs
                "A_id": A_id,
                "S_id": S_id,
                "P_id": P_id,
                "hashtag": hashtag,
                "label": 1,

                # Interaction
                "A_follows_S": int(S_id in A.get("follows_authors", [])),
                "A_mentions_S": A_m_S,
                "A_mentions_S_per": A_m_S_per,
                "S_mentions_A": S_m_A,
                "S_mentions_A_per": S_m_A_per,

                # New interaction features
                "A_reposts_from_S_before_P":
                    reposts_from_author_before(A["history"], S_id, P_time),
                "S_followers_minus_A_followers":
                    S["stats"]["followers"] - A["stats"]["followers"],
                "A_active_before_P":
                    int(parse_dt(A["profile"]["created_at"]) < P_time)
                    if P_time and A["profile"].get("created_at") else 0,

                # User features
                **{f"A_{k}": v for k, v in A_feat.items()},
                **{f"S_{k}": v for k, v in S_feat.items()},
            })

            # ================= NEGATIVES =================
            neg_pool = list(candidate_users - reposted_by)
            if not neg_pool:
                continue

            negs = rng.choice(
                neg_pool,
                size=min(neg_per_pos, len(neg_pool)),
                replace=False,
            )

            for neg_A in negs:
                A = users[neg_A]
                A_feat = user_features(A, max_posts)

                rows.append({
                    # IDs
                    "A_id": neg_A,
                    "S_id": S_id,
                    "P_id": P_id,
                    "hashtag": hashtag,
                    "label": 0,

                    # Interaction
                    "A_follows_S": int(S_id in A.get("follows_authors", [])),
                    "A_mentions_S": 0,
                    "A_mentions_S_per": 0.0,
                    "S_mentions_A": 0,
                    "S_mentions_A_per": 0.0,

                    # New interaction features
                    "A_reposts_from_S_before_P":
                        reposts_from_author_before(A["history"], S_id, P_time),
                    "S_followers_minus_A_followers":
                        S["stats"]["followers"] - A["stats"]["followers"],
                    "A_active_before_P":
                        int(parse_dt(A["profile"]["created_at"]) < P_time)
                        if P_time and A["profile"].get("created_at") else 0,

                    # User features
                    **{f"A_{k}": v for k, v in A_feat.items()},
                    **{f"S_{k}": v for k, v in S_feat.items()},
                })

    df = pd.DataFrame(rows)
    return df


In [21]:
df = build_repost_triples_df(
    raw_user_data,
    post_dict,
    n_posts_sample=10000,  
    neg_per_pos=5
)

print(df.head())
print(df.shape)


                               A_id                              S_id  \
0  did:plc:zwasdkrjhhe7quwhyambron4  did:plc:aun2t4uf5vcw4intgxezrkw3   
1  did:plc:oqqeaqhjkgyde2u7b6vmxfj4  did:plc:aun2t4uf5vcw4intgxezrkw3   
2  did:plc:rjuc43mjd63ogxmolo7x2oio  did:plc:aun2t4uf5vcw4intgxezrkw3   
3  did:plc:6vf2qhhxz3aggohc4wr7e64l  did:plc:aun2t4uf5vcw4intgxezrkw3   
4  did:plc:xjj3lzyw47rztmf72nq4duba  did:plc:aun2t4uf5vcw4intgxezrkw3   

                                                P_id      hashtag  label  \
0  at://did:plc:aun2t4uf5vcw4intgxezrkw3/app.bsky...  Minneapolis      1   
1  at://did:plc:aun2t4uf5vcw4intgxezrkw3/app.bsky...  Minneapolis      0   
2  at://did:plc:aun2t4uf5vcw4intgxezrkw3/app.bsky...  Minneapolis      0   
3  at://did:plc:aun2t4uf5vcw4intgxezrkw3/app.bsky...  Minneapolis      0   
4  at://did:plc:aun2t4uf5vcw4intgxezrkw3/app.bsky...  Minneapolis      0   

   A_follows_S  A_mentions_S  A_mentions_S_per  S_mentions_A  \
0            0             0            

In [23]:
df.to_csv("clean_data.csv", index=False)

In [22]:
def count_positive_entries(df, cols):
    """
    Return a Series with, for each column in `cols`,
    the number of rows where the value is > 0.
    """
    return (df[cols] > 0).sum()

interaction_cols = [
    "A_follows_S",
    "A_mentions_S",
    "A_mentions_S_per",
    "S_mentions_A",
    "S_mentions_A_per",
]

counts = count_positive_entries(df, interaction_cols)
print(counts)


A_follows_S         59
A_mentions_S         2
A_mentions_S_per     2
S_mentions_A         0
S_mentions_A_per     0
dtype: int64
